# 실습: Ollama 파라미터와 Modelfile

Ollama의 생성 옵션과 Modelfile을 사용해 로컬 모델의 답변 스타일을 제어합니다.


---
## 실습 전 주의

- 이 노트북은 `OLLAMA_MODEL`이 설치되어 있어야 합니다.
- 옵션 이름은 OpenAI/Groq/Gemini와 다를 수 있습니다.
- 로컬 모델은 같은 옵션이라도 모델 크기와 계열에 따라 반응이 다릅니다.
---


In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from ollama import Client

load_dotenv()

HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
MODEL = os.getenv("OLLAMA_MODEL", "gemma3:1b")
client = Client(host=HOST)

print(f"Ollama host: {HOST}")
print(f"사용 모델: {MODEL}")


Ollama host: http://localhost:11434
사용 모델: gemma4:e2b


In [2]:
def show_metrics(response) -> None:
    print(
        "\n지표: "
        f"prompt_eval_count={getattr(response, 'prompt_eval_count', None)}, "
        f"eval_count={getattr(response, 'eval_count', None)}, "
        f"total_duration={getattr(response, 'total_duration', None)}"
    )


def response_text(response) -> str:
    message = getattr(response, "message", None)
    if message is not None:
        return getattr(message, "content", "") or ""
    return getattr(response, "response", "") or ""


def print_response(response) -> None:
    text = response_text(response)
    if text:
        print(text)
    else:
        print("답변 본문이 비어 있습니다. think=False 또는 num_predict 값을 확인하세요.")

def generate(label: str, prompt: str, system: str | None = None, options: dict | None = None):
    print(f"\n--- {label} ---")
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    try:
        response = client.chat(
            model=MODEL,
            think=False,
            messages=messages,
            options=options or {},
        )
    except Exception as exc:
        print("Ollama 호출 실패:", type(exc).__name__)
        print(exc)
        return None

    print_response(response)
    show_metrics(response)
    return response


---
## 1. 기준 프롬프트


In [3]:
base_prompt = (
    "초보 개발자가 함수형 프로그래밍을 이해할 수 있도록 "
    "핵심 개념과 Python 예시를 설명해줘."
)

baseline = generate(
    "기준 응답",
    base_prompt,
    options={"num_predict": 260},
)



--- 기준 응답 ---
## 초보 개발자를 위한 함수형 프로그래밍(Functional Programming, FP) 입문

안녕하세요! 함수형 프로그래밍(FP)은 코드를 바라보는 **새로운 사고방식**입니다. 기존의 프로그래밍 방식(어떻게 계산할지를 순서대로 지시하는 방식)에서 벗어나, **"무엇을 달성하고 싶은가?"에 집중**하여 문제를 해결하는 패러다임이라고 이해하시면 쉽습니다.

함수형 프로그래밍의 핵심은 **"데이터를 변화시키지 않고(불변성), 함수를 조합하여 복잡한 문제를 해결한다"**는 것입니다.

---

### 1. 함수형 프로그래밍의 핵심 3가지 개념

함수형 프로그래밍을 이해하기 위해 꼭 알아야 할 세 가지 핵심 개념을 소개하겠습니다.

#### 1. 순수 함수 (Pure Functions)

가장 중요한 개념입니다. **순수 함수는 다음 두 가지 조건을 만족해야 합니다.**

1. **동일한 입력에 대해 항상 동일한 출력:** 같은 입력(인자)을 주면, 언제 실행되든 항상 똑같은 결과만 반환해야 합니다.
2. **부수 효과(Side Effect) 없음:**

지표: prompt_eval_count=34, eval_count=260, total_duration=16174209500


---
## 2. `num_predict`: 출력 길이 제한


In [4]:
for limit in [60, 160, 320]:
    generate(
        f"num_predict={limit}",
        base_prompt,
        options={"num_predict": limit},
    )



--- num_predict=60 ---
## 초보 개발자를 위한 함수형 프로그래밍 (Functional Programming, FP) 완벽 가이드

안녕하세요! 함수형 프로그래밍(FP)은 코드를 바라보는 **새로운 관점**이며, 특히 복잡한 문제를 다룰 때 훨씬 **체계적이고 이해

지표: prompt_eval_count=34, eval_count=60, total_duration=4925569200

--- num_predict=160 ---
## 초보 개발자를 위한 함수형 프로그래밍 (Functional Programming, FP) 입문

함수형 프로그래밍(FP)은 **"상태를 변경하지 않고, 수학적인 함수(Function)를 사용하여 문제를 해결하는 프로그래밍 패러다임"**입니다. 기존의 절차 지향(Imperative) 방식과는 사고방식 자체가 다릅니다.

초보 개발자분들이 낯설어 할 수 있는 개념들을 최대한 쉽고 재미있는 Python 예제를 통해 설명해 드리겠습니다!

---

### 1. 핵심 개념 1: 순수 함수 (Pure Functions)

함수형 프로그래밍의 가장 중요한 기반입니다. **순수 함수란, "동일한 입력에 대해서는 항상 동일한

지표: prompt_eval_count=34, eval_count=160, total_duration=10740581100

--- num_predict=320 ---
## 초보 개발자를 위한 함수형 프로그래밍 (Functional Programming, FP) 이해하기

안녕하세요! 함수형 프로그래밍(FP)은 코드를 바라보는 **새로운 관점**입니다. 기존의 절차 중심적인 프로그래밍(어떻게 해야 할지 단계별로 지시하는 방식)과 달리, FP는 **"무엇을 계산할 것인가?"**에 초점을 맞춥니다.

함수형 프로그래밍이 처음에는 조금 낯설게 느껴질 수 있지만, 그 핵심 아이디어는 매우 직관적이며, 코드를 더 **정확하고, 예측 가능하며, 관리하기 쉽게** 만들어줍니다.

---

### 1. 함수형 프로그래

---
## 3. `temperature`: 무작위성 조절

낮을수록 안정적이고 비슷한 답변이 나오기 쉽습니다.
높을수록 표현이 다양해질 수 있습니다.


In [5]:
creative_prompt = "데이터 분석가를 위한 Python 학습 슬로건을 3개 만들어줘."

for temperature in [0.1, 0.7, 1.2]:
    generate(
        f"temperature={temperature}",
        creative_prompt,
        options={"temperature": temperature, "num_predict": 180},
    )



--- temperature=0.1 ---
## 데이터 분석가를 위한 Python 학습 슬로건 3가지

다음은 데이터 분석가를 위한 Python 학습에 동기를 부여하고 목표를 명확하게 해줄 수 있는 세 가지 슬로건입니다.

**1. 실용성과 강력함을 강조:**

> **"Python으로 데이터를 읽고, 분석하고, 통찰을 발견하라: 당신의 데이터 여정을 시작하세요!"**
> *(강조점: Python이 단순한 언어가 아니라 실제 데이터 분석 과정 전체를 수행하는 강력한 도구임을 강조합니다.)*

**2. 성장과 잠재력을 강조:**

> **"코드를 넘어 인사이트로: Python, 데이터 분석가의 미래를 여는 열쇠!"**
> *(강조점: 코딩 기술을 넘어 최종 목표인 '인사이트(통찰)' 발견에 초점을 맞추어 학습의 궁

지표: prompt_eval_count=25, eval_count=180, total_duration=11781941600

--- temperature=0.7 ---
## 데이터 분석가를 위한 Python 학습 슬로건 3가지

다음은 데이터 분석가를 위한 Python 학습의 중요성과 흥미를 강조하는 세 가지 슬로건입니다.

**1. 실용성과 강력함을 강조:**

> **"Python, 데이터를 현실로 만드는 가장 강력한 도구!"**
> (Python이 데이터를 실제 의미 있는 결과물로 만들어내는 가장 강력한 도구임을 강조하며, 학습의 실용적인 가치를 부각합니다.)

**2. 과정의 몰입감과 성장을 강조:**

> **"코드를 넘어 인사이트를, Python으로 데이터의 비밀을 해독하라!"**
> (단순히 코딩을 배우는 것을 넘어, 그 코드를 통해 숨겨진 통찰(인사이트)을 발견하고 데이터를 해석하는 과정에 초점을 맞춥니다.)

**3.

지표: prompt_eval_count=25, eval_count=180, total_duration=11599318500

--- temperature=1.2 ---
## 데이터 분석가를 위한 Python 학습 슬

---
## 4. `top_p`, `top_k`: 후보 토큰 범위 조절


In [6]:
settings = [
    {"top_p": 0.4, "top_k": 20},
    {"top_p": 0.8, "top_k": 40},
    {"top_p": 1.0, "top_k": 100},
]

for options in settings:
    generate(
        f"top_p={options['top_p']}, top_k={options['top_k']}",
        creative_prompt,
        options={**options, "num_predict": 180},
    )



--- top_p=0.4, top_k=20 ---
## 데이터 분석가를 위한 Python 학습 슬로건 3가지

다음은 데이터 분석가를 위한 Python 학습에 동기를 부여하고 목표를 명확히 해줄 수 있는 세 가지 슬로건입니다.

**1. 실용성과 강력함을 강조하는 슬로건:**

> **"Python으로 데이터를 읽고, 분석하고, 통찰을 발견하라!"**
> *(Why: Python이 데이터 처리의 핵심 도구이며, 단순한 코딩을 넘어 실제 가치(통찰)를 얻는 데 초점을 맞춥니다.)*

**2. 성장과 잠재력을 강조하는 슬로건:**

> **"Python 마스터, 데이터 스토리텔러로 거듭나다!"**
> *(Why: 단순히 코드를 아는 것을 넘어, 분석 결과를 효과적으로 설명하고 전달하는 '스토리텔링' 능력까지

지표: prompt_eval_count=25, eval_count=180, total_duration=11703943600

--- top_p=0.8, top_k=40 ---
## 데이터 분석가를 위한 Python 학습 슬로건 3가지

다음은 데이터 분석가 지망생들을 위한 동기 부여와 목표를 제시하는 세 가지 슬로건입니다. 각 슬로건은 강조하는 측면이 조금씩 다릅니다.

---

### 1. 실용성과 능력 강조 (Practicality & Skill Focus)

**"Python으로 데이터를 읽고, 분석하고, 통찰하라. 당신의 데이터가 미래를 바꾼다!"**

* **강조점:** Python이 단순한 언어가 아니라 실제 데이터 처리와 깊은 통찰(Insight)을 얻는 강력한 도구임을 강조합니다. 실용적인 결과에 초점을 맞춥니다.

### 2. 성장과 잠재력 강조 (Growth & Potential Focus)

**"코드를 넘어, 의미를 발견하다. Python으로 데이터

지표: prompt_eval_count=25, eval_count=180, total_duration=11970778400

--- top_p=1.0, top_k=100 ---
## 데이

---
## 5. `seed`: 재현성 확인


In [7]:
seed_prompt = "API 키를 안전하게 관리하는 방법을 한 문장으로 알려줘."

for index in range(2):
    generate(
        f"seed 고정 실행 {index + 1}",
        seed_prompt,
        options={"seed": 42, "temperature": 0.7, "num_predict": 120},
    )



--- seed 고정 실행 1 ---
API 키는 **환경 변수**나 **안전한 비밀 관리 시스템**을 사용하여 코드에 직접 노출되지 않도록 관리해야 합니다.

지표: prompt_eval_count=24, eval_count=32, total_duration=3663958800

--- seed 고정 실행 2 ---
**API 키는 환경 변수나 안전한 비밀 관리 시스템에 저장하여 코드에 직접 노출되지 않도록 관리해야 합니다.**

지표: prompt_eval_count=24, eval_count=29, total_duration=3132776400


---
## 6. `stop`: 특정 문자열에서 출력 중단


In [8]:
stop_prompt = (
    "다음 형식으로 세 단계를 작성해줘. "
    "각 단계 끝에는 ---끝---을 붙여줘. "
    "주제: 로컬 LLM을 안전하게 사용하는 법"
)

generate(
    "stop 없이 실행",
    stop_prompt,
    options={"num_predict": 260},
)

generate(
    "stop=['---끝---'] 적용",
    stop_prompt,
    options={"stop": ["---끝---"], "num_predict": 260},
)



--- stop 없이 실행 ---
## 주제: 로컬 LLM을 안전하게 사용하는 법

**1단계: 신뢰할 수 있는 소스에서 모델 및 데이터 확보하기**
로컬 LLM을 사용하기 위해서는 모델의 출처와 학습 데이터의 안전성을 확인하는 것이 중요합니다. 널리 알려지고 커뮤니티에서 검증된 오픈소스 모델(예: Hugging Face 등)을 사용하고, 해당 모델이 어떤 데이터로 훈련되었는지, 잠재적인 편향이나 유해한 콘텐츠를 포함하고 있지 않은지 신중하게 조사해야 합니다. 불분명하거나 출처가 의심스러운 모델이나 데이터를 사용하는 것은 위험할 수 있습니다.
---끝---

**2단계: 입력 및 출력에 대한 보안 조치 강화하기**
로컬 환경에서 LLM을 사용할 때, 민감한 정보(개인 식별 정보, 기밀 정보 등)를 모델에 입력하지 않도록 주의해야 합니다. 또한, 모델의 출력 결과가 악의적인 프롬프트 인젝션이나 유해한 정보를 생성하지 않도록 안전 장치를 마련해야 합니다. 입력 필터링 메커니즘을 적용하거나, 민감한 데이터 처리를 최소화하는 방식으로 시스템을 설정해야 합니다.
---끝---

지표: prompt_eval_count=42, eval_count=260, total_duration=16769666400

--- stop=['---끝---'] 적용 ---
## 로컬 LLM을 안전하게 사용하는 법

**1단계: 신뢰할 수 있는 모델 및 소스 선택하기**

사용하려는 로컬 LLM이 어떤 데이터로 훈련되었는지, 그리고 개발팀이나 커뮤니티가 투명하게 공개하고 있는지 확인해야 합니다. 출처가 불분명하거나 악의적인 의도가 있을 수 있는 모델은 사용하지 않는 것이 안전합니다. 오픈 소스 허가(라이선스) 조건도 반드시 검토하여 상업적 또는 개인적 사용에 제약이 없는지 확인하세요.


지표: prompt_eval_count=42, eval_count=123, total_duration=8461342400


ChatResponse(model='gemma4:e2b', created_at='2026-07-03T09:40:02.4104952Z', done=True, done_reason='stop', total_duration=8461342400, load_duration=1629215900, prompt_eval_count=42, prompt_eval_duration=113258000, eval_count=123, eval_duration=6704788000, message=Message(role='assistant', content='## 로컬 LLM을 안전하게 사용하는 법\n\n**1단계: 신뢰할 수 있는 모델 및 소스 선택하기**\n\n사용하려는 로컬 LLM이 어떤 데이터로 훈련되었는지, 그리고 개발팀이나 커뮤니티가 투명하게 공개하고 있는지 확인해야 합니다. 출처가 불분명하거나 악의적인 의도가 있을 수 있는 모델은 사용하지 않는 것이 안전합니다. 오픈 소스 허가(라이선스) 조건도 반드시 검토하여 상업적 또는 개인적 사용에 제약이 없는지 확인하세요.\n', thinking=None, images=None, tool_name=None, tool_calls=None), logprobs=None)

---
## 7. 시스템 메시지와 사용자 메시지 분리


In [9]:
prompt = "클래스와 객체의 차이를 알려줘."

generate(
    "일반 설명",
    prompt,
    options={"num_predict": 220},
)

generate(
    "면접 대비 설명",
    prompt,
    system=(
        "당신은 주니어 개발자 면접 코치입니다. "
        "정의, 비유, 면접 답변 예시 순서로 작성하세요."
    ),
    options={"num_predict": 340},
)



--- 일반 설명 ---
## 클래스와 객체의 차이점

**클래스(Class)**와 **객체(Object)**는 **객체 지향 프로그래밍(Object-Oriented Programming, OOP)**의 가장 기본적인 두 가지 핵심 개념입니다. 쉽게 비유를 들어서 이 둘의 차이점을 설명해 드릴게요.

---

### 1. 클래스 (Class): 설계도 또는 틀 (Blueprint)

**클래스는 객체를 만들기 위한 '설계도', '틀', 또는 '추상적인 개념'**을 의미합니다. 클래스 자체는 메모리에 데이터를 저장하지 않으며, **데이터의 구조와 동작(메서드)**을 정의하는 **템플릿** 역할을 합니다.

**비유:** **집 설계도**

* **특징:**
    * **추상적:** 실제 존재하는 실체라기보다는 어떤 것을 어떻게 만들지 정의하는 개념입니다.
    * **구조 정의:** 데이터(속성/변수)와 그 데이터를 처리하는 함수(메서드/기능)

지표: prompt_eval_count=19, eval_count=220, total_duration=14033101100

--- 면접 대비 설명 ---
## 클래스와 객체의 차이점 (주니어 개발자 면접 대비)

안녕하세요! 주니어 개발자 면접에서 **"클래스와 객체의 차이"**는 가장 기본적이면서도 핵심적인 질문 중 하나입니다. 단순히 정의를 아는 것을 넘어, 그 개념을 실제 코드로 설명하고 비유를 통해 명확하게 전달하는 것이 중요합니다.

제가 면접 코치로서 이 주제를 어떻게 정리하고 답변해야 하는지 **정의, 비유, 면접 답변 예시** 순서로 단계별로 안내해 드리겠습니다.

---

### 1. 정의 (Definition)

가장 먼저 정확한 기술적 정의를 명확히 이해해야 합니다.

*   **클래스 (Class):** 객체를 만들기 위한 **설계도(Blueprint)** 또는 **틀(Template)**입니다. 클래스는 데이터(속성/Attribute)와 그 데이터를 조작하는 기능(메서드/Method)을 묶

ChatResponse(model='gemma4:e2b', created_at='2026-07-03T09:40:37.8467328Z', done=True, done_reason='length', total_duration=21387106300, load_duration=1597238300, prompt_eval_count=53, prompt_eval_duration=424036000, eval_count=340, eval_duration=19354243000, message=Message(role='assistant', content='## 클래스와 객체의 차이점 (주니어 개발자 면접 대비)\n\n안녕하세요! 주니어 개발자 면접에서 **"클래스와 객체의 차이"**는 가장 기본적이면서도 핵심적인 질문 중 하나입니다. 단순히 정의를 아는 것을 넘어, 그 개념을 실제 코드로 설명하고 비유를 통해 명확하게 전달하는 것이 중요합니다.\n\n제가 면접 코치로서 이 주제를 어떻게 정리하고 답변해야 하는지 **정의, 비유, 면접 답변 예시** 순서로 단계별로 안내해 드리겠습니다.\n\n---\n\n### 1. 정의 (Definition)\n\n가장 먼저 정확한 기술적 정의를 명확히 이해해야 합니다.\n\n*   **클래스 (Class):** 객체를 만들기 위한 **설계도(Blueprint)** 또는 **틀(Template)**입니다. 클래스는 데이터(속성/Attribute)와 그 데이터를 조작하는 기능(메서드/Method)을 묶어 놓은 추상적인 개념입니다. 클래스 자체는 메모리에 실제 저장되는 데이터 공간을 차지하지 않습니다.\n*   **객체 (Object):** 클래스라는 설계도를 바탕으로 **실제로 메모리에 생성된 실체(Instance)**입니다. 객체는 클래스에 정의된 속성들을 가지며, 그 속성들을 기반으로 특정 상태와 행동을 가집니다.\n\n**핵심 요약:**\n> 클래스는 \'무엇을 만들지\'를 정의하고, 객체는 그 설계도로 \'실제로 만들어낸 결과물\'입니다.\

---
## 8. Modelfile 확인


In [10]:
modelfile_path = Path("Modelfile.python-tutor")

if modelfile_path.exists():
    print(modelfile_path.read_text(encoding="utf-8"))
else:
    print("Modelfile.python-tutor 파일이 없습니다.")


FROM gemma4:e2b

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER num_ctx 4096

SYSTEM """
당신은 친절한 한국어 Python 선생님입니다.
답변은 쉬운 비유, 짧은 코드, 확인 질문 순서로 작성하세요.
초보자가 헷갈리기 쉬운 부분은 한 문장으로 짚어주세요.
"""



---
## [9. 커스텀 모델 만들기](커스텀%20모델%20만들기(Modelfile).pdf)


In [11]:
# 터미널에서 실행하세요.
# ollama create python-tutor -f Modelfile.python-tutor
# ollama run python-tutor

print("터미널 명령:")
print("ollama create python-tutor -f Modelfile.python-tutor")
print("ollama run python-tutor")


터미널 명령:
ollama create python-tutor -f Modelfile.python-tutor
ollama run python-tutor


---
## 10. 커스텀 모델 호출


In [12]:
# 위 명령으로 python-tutor 모델을 만든 뒤 실행하세요.

CUSTOM_MODEL = "python-tutor"
RUN_CUSTOM_MODEL = False

if RUN_CUSTOM_MODEL:
    response = client.chat(
        model=CUSTOM_MODEL,
        think=False,
        messages=[{"role": "user", "content": "리스트와 튜플의 차이를 알려줘."}],
        options={"num_predict": 280},
    )
    print_response(response)
else:
    print("RUN_CUSTOM_MODEL=True로 바꾸면 커스텀 모델을 호출합니다.")


RUN_CUSTOM_MODEL=True로 바꾸면 커스텀 모델을 호출합니다.


---
## 11. 실무 레시피


In [13]:
# 레시피 1: 로컬 요약
generate(
    "로컬 요약",
    "Ollama를 수업에서 사용할 때 장점 3개와 주의점 3개를 정리해줘.",
    system="표 형식으로 간결하게 작성하세요.",
    options={"temperature": 0.2, "num_predict": 260},
)



--- 로컬 요약 ---
## Ollama를 수업에서 사용할 때의 장점 및 주의점

| 구분 | 내용 | 상세 설명 |
| :--- | :--- | :--- |
| **장점 1** | **설치 및 사용의 용이성 (Ease of Use)** | Docker와 유사하게 간단한 명령어(`ollama run [모델명]`)만으로 로컬 환경에 LLM을 쉽게 설치하고 실행할 수 있어, 복잡한 환경 설정 없이 바로 실습에 집중할 수 있습니다. |
| **장점 2** | **로컬 환경에서의 학습 및 실험 (Local Experimentation)** | 인터넷 연결 없이도(또는 제한적으로) 모델을 구동하여 데이터 보안이 중요한 환경에서 안전하게 LLM의 작동 방식, 프롬프트 엔지니어링, 모델의 한계 등을 직접 실험하고 탐구할 수 있습니다. |
| **장점 3** | **접근성과 비용 효율성 (Accessibility & Cost-Effectiveness)** | 고가의 API 사용료 없이 개인 컴퓨터 자원을 활용하여 다양한 오픈소스 모델(Llama, Mistral 등)을 무료로 구동하고 수업에 적용할 수 있어 교육 예산 측면에서 매우 효율적입니다. |
| **주의점

지표: prompt_eval_count=45, eval_count=260, total_duration=16796972800


ChatResponse(model='gemma4:e2b', created_at='2026-07-03T09:40:54.7027814Z', done=True, done_reason='length', total_duration=16796972800, load_duration=1689421200, prompt_eval_count=45, prompt_eval_duration=409572000, eval_count=260, eval_duration=14684686000, message=Message(role='assistant', content='## Ollama를 수업에서 사용할 때의 장점 및 주의점\n\n| 구분 | 내용 | 상세 설명 |\n| :--- | :--- | :--- |\n| **장점 1** | **설치 및 사용의 용이성 (Ease of Use)** | Docker와 유사하게 간단한 명령어(`ollama run [모델명]`)만으로 로컬 환경에 LLM을 쉽게 설치하고 실행할 수 있어, 복잡한 환경 설정 없이 바로 실습에 집중할 수 있습니다. |\n| **장점 2** | **로컬 환경에서의 학습 및 실험 (Local Experimentation)** | 인터넷 연결 없이도(또는 제한적으로) 모델을 구동하여 데이터 보안이 중요한 환경에서 안전하게 LLM의 작동 방식, 프롬프트 엔지니어링, 모델의 한계 등을 직접 실험하고 탐구할 수 있습니다. |\n| **장점 3** | **접근성과 비용 효율성 (Accessibility & Cost-Effectiveness)** | 고가의 API 사용료 없이 개인 컴퓨터 자원을 활용하여 다양한 오픈소스 모델(Llama, Mistral 등)을 무료로 구동하고 수업에 적용할 수 있어 교육 예산 측면에서 매우 효율적입니다. |\n| **주의점', thinking=None, images=None, tool_name=None, tool_calls=None), logprobs=None)

In [14]:
# 레시피 2: 코드 생성
generate(
    "코드 생성",
    "Python requests로 http://localhost:11434/api/tags를 호출하는 예제를 작성해줘.",
    system="코드는 하나의 블록으로 작성하고, 아래에 설명 2문장을 붙이세요.",
    options={"temperature": 0.4, "num_predict": 340},
)



--- 코드 생성 ---
```python
import requests

url = "http://localhost:11434/api/tags"
response = requests.get(url)

print(response.status_code)
print(response.json())
```
이 코드는 `requests` 라이브러리를 사용하여 지정된 URL로 GET 요청을 보냅니다. 응답 상태 코드와 JSON 형식의 데이터를 출력하여 API 호출 결과를 확인합니다.

지표: prompt_eval_count=59, eval_count=91, total_duration=7156638000


ChatResponse(model='gemma4:e2b', created_at='2026-07-03T09:41:01.8729775Z', done=True, done_reason='stop', total_duration=7156638000, load_duration=1548490800, prompt_eval_count=59, prompt_eval_duration=473221000, eval_count=91, eval_duration=5120367000, message=Message(role='assistant', content='```python\nimport requests\n\nurl = "http://localhost:11434/api/tags"\nresponse = requests.get(url)\n\nprint(response.status_code)\nprint(response.json())\n```\n이 코드는 `requests` 라이브러리를 사용하여 지정된 URL로 GET 요청을 보냅니다. 응답 상태 코드와 JSON 형식의 데이터를 출력하여 API 호출 결과를 확인합니다.', thinking=None, images=None, tool_name=None, tool_calls=None), logprobs=None)

In [15]:
# 레시피 3: 창의적 글쓰기
generate(
    "창의적 글쓰기",
    "로컬 LLM 수업을 홍보하는 짧은 문구 5개를 만들어줘.",
    options={"temperature": 1.1, "top_p": 0.95, "num_predict": 240},
)



--- 창의적 글쓰기 ---
## 로컬 LLM 수업 홍보 문구 5가지 ✨

다음은 로컬 LLM 수업을 홍보하기 위한 매력적인 문구 5가지입니다. 타겟층과 강조하고 싶은 부분에 따라 선택하여 사용해 보세요! 😊

**1. 실습 및 통제 강조 (기술 중심)**

> **"클라우드 의존 없이, 내 컴퓨터에서 LLM을 직접 구동하는 마법! 🧙‍♂️ 로컬 LLM의 모든 것을 직접 코딩하며 이해하세요."**
> *(핵심: 독립성, 실습 중심)*

**2. 개인화 및 프라이버시 강조 (개인 사용자 타겟)**

> **"당신의 데이터는 안전하게! 🔒 민감 정보 걱정 없이 나만의 맞춤형 AI 모델을 구축하는 로컬 LLM 마스터 클래스."**
> *(핵심: 보안, 개인화)*

**3. 최신 트렌드 및 접근성 강조 (트렌드 추종자 타겟)**

> **"ChatGPT를 넘어, 로컬 LLM 시대의 핵심 열쇠! 🔥 최신 기술 트렌드를

지표: prompt_eval_count=28, eval_count=240, total_duration=15481704400


ChatResponse(model='gemma4:e2b', created_at='2026-07-03T09:41:17.3712967Z', done=True, done_reason='length', total_duration=15481704400, load_duration=1601409100, prompt_eval_count=28, prompt_eval_duration=264267000, eval_count=240, eval_duration=13603484000, message=Message(role='assistant', content='## 로컬 LLM 수업 홍보 문구 5가지 ✨\n\n다음은 로컬 LLM 수업을 홍보하기 위한 매력적인 문구 5가지입니다. 타겟층과 강조하고 싶은 부분에 따라 선택하여 사용해 보세요! 😊\n\n**1. 실습 및 통제 강조 (기술 중심)**\n\n> **"클라우드 의존 없이, 내 컴퓨터에서 LLM을 직접 구동하는 마법! 🧙\u200d♂️ 로컬 LLM의 모든 것을 직접 코딩하며 이해하세요."**\n> *(핵심: 독립성, 실습 중심)*\n\n**2. 개인화 및 프라이버시 강조 (개인 사용자 타겟)**\n\n> **"당신의 데이터는 안전하게! 🔒 민감 정보 걱정 없이 나만의 맞춤형 AI 모델을 구축하는 로컬 LLM 마스터 클래스."**\n> *(핵심: 보안, 개인화)*\n\n**3. 최신 트렌드 및 접근성 강조 (트렌드 추종자 타겟)**\n\n> **"ChatGPT를 넘어, 로컬 LLM 시대의 핵심 열쇠! 🔥 최신 기술 트렌드를', thinking=None, images=None, tool_name=None, tool_calls=None), logprobs=None)